# Minimal Encoder -> ODE -> Decoder Demo

Uses `results/separate_multisine_05_20260318_095907` with the aligned `multisine_05` true-label CSV.

In [ ]:
import json, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import BabVideoPipelineConfig
from src.data.registry import ensure_true_labels
from src.models.base import load_model, resolve_device
from src.vision.datasets import FrameStateDataset, load_bab_with_video
from src.vision.models import DecoderFrameDeconv, EncOdeDecModel, EncoderThetaNet

In [ ]:
run_dir = ROOT / "results/separate_multisine_05_20260318_095907"
cfg = BabVideoPipelineConfig.from_dict(json.loads((run_dir / "config.json").read_text()))
assert cfg.encoder == "theta_regression" and cfg.encoder_velocity_mode == "full_encoder"

true_csv = str(ensure_true_labels(cfg.dataset))
data, frames, frame_idx_map, aux = load_bab_with_video(
    cfg.dataset,
    video_dir=cfg.video_dir,
    video_path=cfg.video_path,
    resample_factor=cfg.resample_factor,
    video_fps=cfg.video_fps,
    frame_height=cfg.frame_height,
    frame_width=cfg.frame_width,
    preprocess=True,
    led_frame=cfg.led_frame,
    use_led_sync=cfg.use_led_sync,
    keypoint_labels_csv=true_csv,
    theta_labels_csv=true_csv,
    align_theta=cfg.align_theta,
    alignment_offset_min_s=cfg.alignment_offset_min_s,
    alignment_offset_max_s=cfg.alignment_offset_max_s,
    auto_match_video_fps=cfg.auto_match_video_fps,
    return_aux=True,
)

device = resolve_device(cfg.device)
encoder = EncoderThetaNet(pretrained=False, state_dim=2, in_channels=6).to(device).eval()
encoder.load_state_dict(torch.load(run_dir / "encoder_best.pt", map_location=device))
decoder = DecoderFrameDeconv(frame_height=cfg.frame_height, frame_width=cfg.frame_width).to(device).eval()
decoder.load_state_dict(torch.load(run_dir / "decoder_image_best.pt", map_location=device))
model = EncOdeDecModel(encoder, load_model(run_dir / "ode_model.pt").ode_func_, decoder, dt=1 / data.sampling_rate, encoder_velocity_mode=cfg.encoder_velocity_mode).to(device).eval()
frame_ds = FrameStateDataset(data, frames, frame_idx_map, encoder_velocity_mode=cfg.encoder_velocity_mode)

print(run_dir.name, "| samples:", len(data), "| frames:", len(frames), "| device:", device)
print("theta alignment:", aux.get("theta_alignment"))

In [ ]:
theta_dot = data.y_dot if data.y_dot is not None else np.gradient(data.y, 1 / data.sampling_rate)
i = int(np.nanargmax(np.abs(theta_dot[:-3])))
x_in, x0_true, meta = frame_ds[i]
frame_ids = frame_idx_map[i : i + 3]
prev_frame, curr_frame = frames[meta["frame_idx_prev"]], frames[meta["frame_idx"]]
u = torch.tensor(np.asarray(data.u[i : i + 3], dtype=np.float32).reshape(-1, 1), device=device)

with torch.no_grad():
    x0_hat = encoder(x_in.unsqueeze(0).to(device))
    x_seq = model.rollout(x0_hat, u, 3)
    pred_frames = model.decode(x_seq)[:, 0].cpu().permute(0, 2, 3, 1).numpy()

pred_states = x_seq[:, 0].cpu().numpy()
true_states = np.c_[data.y[i : i + 3], theta_dot[i : i + 3]]

print("full_encoder input = [current frame, current - previous frame]")
print("t =", round(float(data.t[i]), 3), "s | x0_hat =", x0_hat.cpu().numpy().round(3), "| x0_true =", x0_true.numpy().round(3))
print("pred states:\n", pred_states.round(3))
print("true states:\n", true_states.round(3))

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
for a, img, title in zip(ax, [prev_frame, curr_frame], [f"previous frame ({meta['frame_idx_prev']})", f"current frame ({meta['frame_idx']})"]):
    a.imshow(img); a.set_title(title); a.axis("off")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(2, 3, figsize=(10, 5))
for k in range(3):
    ax[0, k].imshow(frames[frame_ids[k]]); ax[0, k].set_title(f"true t+{k}"); ax[0, k].axis("off")
    ax[1, k].imshow(np.clip(pred_frames[k], 0, 1)); ax[1, k].set_title(f"decoded t+{k}"); ax[1, k].axis("off")
fig.suptitle("Top: aligned video frames | Bottom: encoder -> ODE -> decoder", y=0.98)
plt.tight_layout(); plt.show()